# 🟫🟩 Caderno 04: Transformação Bronze → Silver (Camadas de Qualidade)

## 🎯 1. Objetivo do Notebook

Este caderno documenta e executa a **transformação de dados brutos (Bronze) em dados validados e enriquecidos (Silver)**. O foco é:

1. **Carregar** 122.543 incidentes brutos do ITSM (dados como chegam)
2. **Filtrar** apenas esforço real (Status != 'Sem Intervenção') e dados recentes (>= 2025)
3. **Criar** 6 features derivadas para ML
4. **Validar** qualidade dos dados
5. **Persistir** como Silver Parquet em S3

### Fluxo Visual

```
S3 Bronze (ITSM bruto)              S3 Silver (pronto para ML)
├─ incidents_standardized.parquet   ├─ incidents_silver_2025.parquet
│  ├─ 122.543 registros             │  ├─ 41.441 registros
│  ├─ 19 colunas originais          │  ├─ 25 colunas (originais + 6 features)
│  └─ Período: 2018-2026            │  ├─ Período: 2025+
│                                    │  └─ Apenas Status != 'Sem Intervenção'
│
└─→ [Notebook 04: Transform] ────→ ├─ Pronto para E6 (dbt Silver)
                                    └─ Pronto para E7 (Gold + ML)
```

---

## 📚 2. Dicionário de Dados - Entrada Bronze

### Colunas Originais (19 campos do ITSM)

| # | Coluna | Tipo | Origem | Descrição |
|---|--------|------|--------|----------|
| 0 | `Número` | string | ITSM | ID único do incidente (ex: INC0028941) |
| 1 | `Prioridade` | string | ITSM | P1-P5 (ex: "1 - Crítica", "3 - Moderada") |
| 2 | `Produto` | string | ITSM | Sistema/aplicação impactada (pode ser nulo) |
| 3 | `Categoria` | string | ITSM | Tipo de falha (Acesso, Hardware, Software, nulo) |
| 4 | `Subcategoria` | string | ITSM | Detalhe da categoria (pode ser nulo) |
| 5 | `Grupo_designado` | string | ITSM | Equipe responsável |
| 6 | `Aberto` | datetime | ITSM | Timestamp de abertura |
| 7 | `Resolvido` | datetime | ITSM | Timestamp de resolução (pode ser nulo) |
| 8 | `Encerrado` | datetime | ITSM | Timestamp de encerramento |
| 9 | `Duração` | int | ITSM | Segundos entre abertura e encerramento |
| 10 | `Status` | string | ITSM | "Aberto", "Resolvido", "Encerrado", "Sem Intervenção" |
| 11 | `Entrou_para_KPI` | string | ITSM | "SIM" ou "NAO" |
| 12 | `KPI_Violado` | string | ITSM | "SIM" ou "NAO" (pode ser nulo) - **TARGET BRUTO** |
| 13 | `Incidente_Pai` | string | ITSM | ID do incidente pai (pode ser nulo) |
| 14 | `Código_de_fechamento` | string | ITSM | Motivo do encerramento (pode ser nulo) |
| 15 | `Solução` | string | ITSM | Descrição da solução (pode ser nulo) |
| 16 | `Aberto_por` | string | ITSM | Usuário que abriu |
| 17 | `Descrição_resumida` | string | ITSM | Summary do incidente |

---

## 🔧 3. Filtros e Decisões de Transformação

### Filtro 1: Esforço Real (Status != 'Sem Intervenção')

**Problema**: ITSM registra alertas automáticos que não requerem ação humana.  
**Solução**: Manter apenas `Status != 'Sem Intervenção'`

```
Bronze (122.543 incidentes)
  └─ Status = 'Sem Intervenção' (~81k) → DESCARTADOS (ruído)
  └─ Status ≠ 'Sem Intervenção' (~41k) → MANTIDOS (esforço real)
```

### Filtro 2: Dados Recentes (Data >= 2025-01-01)

**Problema**: Contexto operacional mudou (2018-2024 é história, 2025+ é realidade atual).  
**Solução**: Manter apenas registros a partir de 2025

```
Pós-filtro 1 (~41k incidentes)
  └─ Data < 2025-01-01 → DESCARTADOS (dados históricos irrelevantes)
  └─ Data >= 2025-01-01 → MANTIDOS (comportamento atual)
```

**Resultado Final: 41.441 registros de esforço real, 2025+**

---

## 🔨 4. Features Engineered (6 Derivadas)

| # | Feature | Tipo | Fórmula/Lógica | Propósito |
|---|---------|------|---|---|
| 1 | `Exige_Intervencao` | int (0/1) | `1 if Status != 'Sem Intervenção' else 0` | Flag de esforço real (redundante pós-filtro, útil para EDA) |
| 2 | `Prioridade_Num` | int (1-5) | `int(Prioridade[0])` | Extrai número P1→1, P5→5 |
| 3 | `Possui_Pai` | int (0/1) | `1 if Incidente_Pai is not null else 0` | Flag de hierarquia (incidente filho) |
| 4 | `Duracao_Horas` | float | `Duração / 3600` | Converte segundos em horas |
| 5 | `Data_Abertura` | date | `Aberto.date()` | Extrai data sem hora |
| 6 | `KPI_Status_Int` | int (-1/0/1) | `{SIM:1, NAO:0, null:-1}` | Codifica KPI_Violado com sentinela para nulos |

---

## 🧹 5. Tratamento de Nulos (Regras de Negócio)

| Coluna | Nulo Original | Valor Preenchido | Justificativa |
|--------|---|---|---|
| `Produto` | ✓ | `'Não Classificado'` | Sistema desconhecido/genérico |
| `Categoria` | ✓ | `'Não Classificado'` | Falha desconhecida |
| `Subcategoria` | ✓ | `'Não Informada'` | Documentação incompleta |
| `Incidente_Pai` | ✓ | `'Independente'` | Não vinculado (é raiz) |
| `Código_de_fechamento` | ✓ | `'Não Encerrado'` | Em progresso, não finalizado |
| `Solução` | ✓ | `'Sem Descrição'` | Resolução sem documentação |
| `Resolvido` | ✓ | **NaT (mantém)** | Incidente aberto, sem resolução |
| `Encerrado` | ✓ | **NaT (mantém)** | Não finalizado administrativamente |

---

## 💻 6. Implementação: Carregamento Bronze

### 6.1 Setup e Imports

In [1]:
# Importações
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

print("✅ Importações concluídas")
print(f"   Pandas: {pd.__version__}")
print(f"   NumPy: {np.__version__}")


✅ Importações concluídas
   Pandas: 3.0.5
   NumPy: 2.5.2


### 6.2 Carregar Bronze do S3

In [2]:
# Configurações — lendo direto do banco fiap (public.incidentes) em vez do S3.
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / '.git').exists() or (p / 'etl').is_dir():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from etl.db import get_engine

MIN_DATE = "2025-01-01"

print("📥 Carregando Bronze do banco fiap (public.incidentes)...")
print("-" * 70)

engine = get_engine()
df_raw = pd.read_sql("SELECT * FROM public.incidentes", engine)

# Adapta para o schema Bronze (18 colunas) que o resto do notebook espera
# (nomes/acentuação do projeto original em AWS).
df_bronze = pd.DataFrame({
    'Número': df_raw['numero'],
    'Prioridade': df_raw['prioridade'],
    'Produto': df_raw['produto'],
    'Categoria': df_raw['categoria'],
    'Subcategoria': df_raw['subcategoria'],
    'Grupo_designado': df_raw['grupo_designado'],
    'Aberto': df_raw['aberto'],
    'Resolvido': df_raw['resolvido'],
    'Encerrado': df_raw['encerrado'],
    # duracao_min (banco) -> segundos (schema original: Duracao_Horas = Duração / 3600)
    'Duração': df_raw['duracao_min'] * 60,
    'Status': df_raw['status'],
    # entrou_kpi/kpi_violado são boolean no banco; o notebook espera 'SIM'/'NAO'
    'Entrou_para_KPI': df_raw['entrou_kpi'].map({True: 'SIM', False: 'NAO'}),
    'KPI_Violado': df_raw['kpi_violado'].map({True: 'SIM', False: 'NAO'}),
    'Incidente_Pai': df_raw['incidente_pai'],
    'Código_de_fechamento': df_raw['codigo_fechamento'],
    'Solução': df_raw['solucao'],
    'Aberto_por': df_raw['aberto_por'],
    'Descrição_resumida': df_raw['descricao_resumida'],
})

print(f"✅ Bronze carregado com sucesso!")
print(f"   Total de registros: {len(df_bronze):,}")
print(f"   Colunas: {df_bronze.shape[1]}")
print(f"   Memória: {df_bronze.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


📥 Carregando Bronze do banco fiap (public.incidentes)...
----------------------------------------------------------------------


✅ Bronze carregado com sucesso!
   Total de registros: 122,543
   Colunas: 18
   Memória: 30.84 MB


### 6.3 Inspeção Inicial Bronze

In [3]:
print("\n🔍 Inspeção Inicial dos Dados Bronze")
print("=" * 70)

# Tipos de dados
print("\n📊 Tipos de Dados:")
print(df_bronze.dtypes)

# Nulos por coluna
print("\n🔴 Nulos por Coluna:")
nulos = df_bronze.isnull().sum()
nulos_pct = 100 * nulos / len(df_bronze)
for col in df_bronze.columns:
    if nulos[col] > 0:
        print(f"   {col:25} {nulos[col]:>7,} ({nulos_pct[col]:>5.2f}%)")

# Distribuição de Status (filtro principal)
print("\n🎯 Distribuição de Status (Filtro Principal):")
status_dist = df_bronze['Status'].value_counts()
for status, count in status_dist.items():
    pct = 100 * count / len(df_bronze)
    print(f"   {status:30} {count:>7,} ({pct:>5.2f}%)")

# Range de datas
print(f"\n📅 Range de Datas:")
print(f"   Primeiro incidente: {df_bronze['Aberto'].min()}")
print(f"   Último incidente:   {df_bronze['Aberto'].max()}")


🔍 Inspeção Inicial dos Dados Bronze

📊 Tipos de Dados:
Número                             str
Prioridade                         str
Produto                            str
Categoria                          str
Subcategoria                       str
Grupo_designado                    str
Aberto                  datetime64[us]
Resolvido               datetime64[us]
Encerrado               datetime64[us]
Duração                          int64
Status                             str
Entrou_para_KPI                    str
KPI_Violado                        str
Incidente_Pai                      str
Código_de_fechamento               str
Solução                            str
Aberto_por                         str
Descrição_resumida                 str
dtype: object

🔴 Nulos por Coluna:
   Produto                    77,935 (63.60%)
   Categoria                  77,721 (63.42%)
   Subcategoria               77,720 (63.42%)
   Resolvido                  82,302 (67.16%)
   KPI_Violado         

---

## 🔄 7. Implementação: Transformações Silver

### 7.1 Filtro 1: Remover Ruído (Status != 'Sem Intervenção')

In [4]:
print("\n📊 FILTRO 1: Removendo Ruído de Monitoramento")
print("=" * 70)
print(f"Antes: {len(df_bronze):,} registros")

# Filtrar
df_silver = df_bronze[df_bronze['Status'] != 'Sem Intervenção'].copy()

removed = len(df_bronze) - len(df_silver)
pct_removed = 100 * removed / len(df_bronze)

print(f"Removidos: {removed:,} registros (Status='Sem Intervenção')")
print(f"Percentual: {pct_removed:.2f}%")
print(f"Depois: {len(df_silver):,} registros (esforço real)")


📊 FILTRO 1: Removendo Ruído de Monitoramento
Antes: 122,543 registros
Removidos: 80,373 registros (Status='Sem Intervenção')
Percentual: 65.59%
Depois: 42,170 registros (esforço real)


### 7.2 Filtro 2: Manter Apenas Dados Recentes (>= 2025-01-01)

In [5]:
print("\n📅 FILTRO 2: Manter Apenas Dados Recentes (2025+)")
print("=" * 70)
print(f"Antes: {len(df_silver):,} registros")

# Converter data se necessário
df_silver['Aberto'] = pd.to_datetime(df_silver['Aberto'])
min_date_dt = pd.to_datetime(MIN_DATE)

# Filtrar
df_silver = df_silver[df_silver['Aberto'] >= min_date_dt].copy()

print(f"Data mínima mantida: {MIN_DATE}")
print(f"Depois: {len(df_silver):,} registros (esforço real 2025+)")

# Estatísticas pós-filtro
print(f"\n📈 Resumo dos Filtros:")
print(f"   Bronze original:      {len(df_bronze):,}")
print(f"   Pós Status:          {len(df_bronze[df_bronze['Status'] != 'Sem Intervenção']):,}")
print(f"   Pós Data 2025+:      {len(df_silver):,}")
print(f"   Taxa de retenção:    {100 * len(df_silver) / len(df_bronze):.2f}%")


📅 FILTRO 2: Manter Apenas Dados Recentes (2025+)
Antes: 42,170 registros


Data mínima mantida: 2025-01-01
Depois: 41,441 registros (esforço real 2025+)

📈 Resumo dos Filtros:
   Bronze original:      122,543
   Pós Status:          42,170
   Pós Data 2025+:      41,441
   Taxa de retenção:    33.82%


### 7.3 Criar 6 Features Derivadas

In [6]:
print("\n🔨 FEATURE ENGINEERING: Criando 6 Derivadas")
print("=" * 70)

# Feature 1: Exige_Intervencao (redundante após filtro, mas útil para análise)
df_silver['Exige_Intervencao'] = (df_silver['Status'] != 'Sem Intervenção').astype(int)
print(f"✅ Feature 1: Exige_Intervencao")
print(f"   Valores únicos: {df_silver['Exige_Intervencao'].unique()}")
print(f"   Distribuição:\n{df_silver['Exige_Intervencao'].value_counts()}")

# Feature 2: Prioridade_Num (extrai número de 'P1', 'P2', etc)
df_silver['Prioridade_Num'] = df_silver['Prioridade'].str[0].astype(int)
print(f"\n✅ Feature 2: Prioridade_Num")
print(f"   Range: {df_silver['Prioridade_Num'].min()} a {df_silver['Prioridade_Num'].max()}")
print(f"   Distribuição:\n{df_silver['Prioridade_Num'].value_counts().sort_index()}")

# Feature 3: Possui_Pai (verifica se Incidente_Pai não é nulo)
df_silver['Possui_Pai'] = df_silver['Incidente_Pai'].notna().astype(int)
print(f"\n✅ Feature 3: Possui_Pai")
com_pai = (df_silver['Possui_Pai'] == 1).sum()
print(f"   Incidentes com pai: {com_pai:,} ({100*com_pai/len(df_silver):.2f}%)")
print(f"   Distribuição:\n{df_silver['Possui_Pai'].value_counts()}")

# Feature 4: Duracao_Horas (converte segundos em horas)
df_silver['Duracao_Horas'] = df_silver['Duração'] / 3600
print(f"\n✅ Feature 4: Duracao_Horas")
print(f"   Mín: {df_silver['Duracao_Horas'].min():.2f}h")
print(f"   Máx: {df_silver['Duracao_Horas'].max():.2f}h")
print(f"   Média: {df_silver['Duracao_Horas'].mean():.2f}h")
print(f"   Mediana: {df_silver['Duracao_Horas'].median():.2f}h")

# Feature 5: Data_Abertura (extrai data sem hora)
df_silver['Data_Abertura'] = df_silver['Aberto'].dt.date
df_silver['Ano_Mes'] = df_silver['Aberto'].dt.strftime('%Y-%m')  # Para particionamento
print(f"\n✅ Feature 5: Data_Abertura")
print(f"   Período: {df_silver['Data_Abertura'].min()} a {df_silver['Data_Abertura'].max()}")
print(f"   Número de dias: {df_silver['Data_Abertura'].nunique()}")

# Feature 6: KPI_Status_Int (codifica com sentinela para nulos)
def encode_kpi(val):
    if pd.isna(val):
        return -1  # Sentinela para desconhecido
    elif val == 'SIM':
        return 1   # Violado
    elif val == 'NAO':
        return 0   # Respeitado
    else:
        return -1

df_silver['KPI_Status_Int'] = df_silver['KPI_Violado'].apply(encode_kpi)
print(f"\n✅ Feature 6: KPI_Status_Int")
print(f"   Valores únicos: {sorted(df_silver['KPI_Status_Int'].unique())}")
print(f"   Distribuição:\n{df_silver['KPI_Status_Int'].value_counts().sort_index()}")
print(f"   (-1 = desconhecido, 0 = respeitado, 1 = violado)")


🔨 FEATURE ENGINEERING: Criando 6 Derivadas
✅ Feature 1: Exige_Intervencao
   Valores únicos: [1]
   Distribuição:
Exige_Intervencao
1    41441
Name: count, dtype: int64



✅ Feature 2: Prioridade_Num
   Range: 1 a 5
   Distribuição:
Prioridade_Num
1        1
2     9383
3    23294
4     8439
5      324
Name: count, dtype: int64

✅ Feature 3: Possui_Pai
   Incidentes com pai: 8,501 (20.51%)
   Distribuição:
Possui_Pai
0    32940
1     8501
Name: count, dtype: int64

✅ Feature 4: Duracao_Horas
   Mín: 0.13h
   Máx: 436529.15h
   Média: 5756.09h
   Mediana: 87.90h



✅ Feature 5: Data_Abertura
   Período: 2025-01-01 a 2025-12-31
   Número de dias: 365

✅ Feature 6: KPI_Status_Int
   Valores únicos: [np.int64(-1), np.int64(0), np.int64(1)]
   Distribuição:
KPI_Status_Int
-1    16285
 0    24918
 1      238
Name: count, dtype: int64
   (-1 = desconhecido, 0 = respeitado, 1 = violado)


### 7.4 Tratamento de Nulos (Regras de Negócio)

In [7]:
print("\n🧹 TRATAMENTO DE NULOS: Regras de Negócio")
print("=" * 70)

# Mapeamento de regras
fillna_rules = {
    'Produto': 'Não Classificado',
    'Categoria': 'Não Classificado',
    'Subcategoria': 'Não Informada',
    'Incidente_Pai': 'Independente',
    'Código_de_fechamento': 'Não Encerrado',
    'Solução': 'Sem Descrição'
}

# Aplicar regras
for col, fill_value in fillna_rules.items():
    nulos_antes = df_silver[col].isnull().sum()
    df_silver[col] = df_silver[col].fillna(fill_value)
    nulos_depois = df_silver[col].isnull().sum()
    print(f"✅ {col:25} {nulos_antes:>5,} nulos → {nulos_depois:>5,} (preenchidos com '{fill_value}')")

print(f"\n⏸️  Mantidos como NaT (datetime nulos):")
print(f"   Resolvido: {df_silver['Resolvido'].isnull().sum():,} nulos")
print(f"   Encerrado: {df_silver['Encerrado'].isnull().sum():,} nulos")
print(f"   (Representam incidentes não finalizados - informação legítima)")


🧹 TRATAMENTO DE NULOS: Regras de Negócio
✅ Produto                   2,149 nulos →     0 (preenchidos com 'Não Classificado')
✅ Categoria                 1,946 nulos →     0 (preenchidos com 'Não Classificado')
✅ Subcategoria              1,945 nulos →     0 (preenchidos com 'Não Informada')
✅ Incidente_Pai             32,940 nulos →     0 (preenchidos com 'Independente')
✅ Código_de_fechamento      1,355 nulos →     0 (preenchidos com 'Não Encerrado')
✅ Solução                   26,241 nulos →     0 (preenchidos com 'Sem Descrição')

⏸️  Mantidos como NaT (datetime nulos):
   Resolvido: 1,862 nulos
   Encerrado: 0 nulos
   (Representam incidentes não finalizados - informação legítima)


### 7.5 Calcular Target: Target_Risco_SLA

In [8]:
print("\n🎯 CÁLCULO DO TARGET: Target_Risco_SLA")
print("=" * 70)

# Inicializar target com KPI_Status_Int
df_silver['Target_Risco_SLA'] = df_silver['KPI_Status_Int']

print(f"\nCamada 1: Base (usar KPI_Violado diretamente)")
base_coverage = (df_silver['KPI_Status_Int'].isin([0, 1])).sum()
print(f"   Cobertura: {base_coverage:,} registros ({100*base_coverage/len(df_silver):.2f}%)")

# Camada 2: Heurística para KPI desconhecido
print(f"\nCamada 2: Heurística (se KPI desconhecido e duração > SLA)")
heuristica_mask = (
    (df_silver['KPI_Status_Int'] == -1) & 
    (df_silver['Prioridade_Num'] == 2) & 
    (df_silver['Duracao_Horas'] > 4)  # P2 SLA = 4h
)
heuristica_count = heuristica_mask.sum()
df_silver.loc[heuristica_mask, 'Target_Risco_SLA'] = 1
print(f"   Corrigidos (P2 + Duração > 4h): {heuristica_count:,} registros")
print(f"   Cobertura acumulada: {base_coverage + heuristica_count:,} ({100*(base_coverage+heuristica_count)/len(df_silver):.2f}%)")

# Camada 3: Isenções
print(f"\nCamada 3: Isenções (incidentes filhos, sem intervenção)")
isencoes = (
    (df_silver['Possui_Pai'] == 1) |
    (df_silver['Exige_Intervencao'] == 0)
)
isencoes_count = isencoes.sum()
df_silver.loc[isencoes, 'Target_Risco_SLA'] = 0
print(f"   Isentos: {isencoes_count:,} registros")

# Distribuição final
print(f"\n📊 Distribuição Final do Target:")
target_dist = df_silver['Target_Risco_SLA'].value_counts().sort_index()
for label, count in target_dist.items():
    pct = 100 * count / len(df_silver)
    print(f"   Target={label}: {count:>7,} registros ({pct:>6.2f}%)")
    if label == 1:
        print(f"   ⚠️  Desbalanceamento: 1 violação para cada {len(df_silver)/count:.0f} conformidades")


🎯 CÁLCULO DO TARGET: Target_Risco_SLA

Camada 1: Base (usar KPI_Violado diretamente)
   Cobertura: 25,156 registros (60.70%)

Camada 2: Heurística (se KPI desconhecido e duração > SLA)
   Corrigidos (P2 + Duração > 4h): 4,153 registros
   Cobertura acumulada: 29,309 (70.72%)

Camada 3: Isenções (incidentes filhos, sem intervenção)
   Isentos: 8,501 registros

📊 Distribuição Final do Target:
   Target=-1:   7,755 registros ( 18.71%)
   Target=0:  33,419 registros ( 80.64%)
   Target=1:     267 registros (  0.64%)
   ⚠️  Desbalanceamento: 1 violação para cada 155 conformidades


---

## ✅ 8. Validações de Qualidade Pós-Processamento

### 8.1 Validação de Schema

In [9]:
print("\n✅ VALIDAÇÃO 1: Schema Esperado")
print("=" * 70)

expected_cols = [
    # Originais Bronze
    'Número', 'Prioridade', 'Produto', 'Categoria', 'Subcategoria',
    'Grupo_designado', 'Aberto', 'Resolvido', 'Encerrado', 'Duração',
    'Status', 'Entrou_para_KPI', 'KPI_Violado', 'Incidente_Pai',
    'Código_de_fechamento', 'Solução', 'Aberto_por', 'Descrição_resumida',
    # Engineered
    'Exige_Intervencao', 'Prioridade_Num', 'Possui_Pai', 'Duracao_Horas',
    'Data_Abertura', 'KPI_Status_Int', 'Target_Risco_SLA', 'Ano_Mes'
]

missing = set(expected_cols) - set(df_silver.columns)
extra = set(df_silver.columns) - set(expected_cols)

print(f"✅ Colunas esperadas presentes: {len(expected_cols)}")
if missing:
    print(f"❌ Colunas faltando: {missing}")
else:
    print(f"✅ Nenhuma coluna faltando")

if extra:
    print(f"⚠️  Colunas extras: {extra}")
else:
    print(f"✅ Nenhuma coluna extra")

print(f"\n📊 Shape final: {df_silver.shape[0]:,} registros × {df_silver.shape[1]} colunas")


✅ VALIDAÇÃO 1: Schema Esperado
✅ Colunas esperadas presentes: 26
✅ Nenhuma coluna faltando
✅ Nenhuma coluna extra

📊 Shape final: 41,441 registros × 26 colunas


### 8.2 Validação de Nulos Críticos

In [10]:
print("\n✅ VALIDAÇÃO 2: Nulos em Colunas Críticas")
print("=" * 70)

colunas_criticas = [
    'Número',
    'Prioridade_Num',
    'Duracao_Horas',
    'Target_Risco_SLA',
    'Data_Abertura'
]

has_critical_nulls = False
for col in colunas_criticas:
    nulos = df_silver[col].isnull().sum()
    if nulos > 0:
        print(f"❌ {col:25} {nulos:>5,} nulos ⚠️")
        has_critical_nulls = True
    else:
        print(f"✅ {col:25} 0 nulos")

if has_critical_nulls:
    print("\n⚠️  ERRO: Colunas críticas contêm nulos!")
else:
    print("\n✅ Nenhuma coluna crítica com nulos")


✅ VALIDAÇÃO 2: Nulos em Colunas Críticas
✅ Número                    0 nulos
✅ Prioridade_Num            0 nulos
✅ Duracao_Horas             0 nulos
✅ Target_Risco_SLA          0 nulos
✅ Data_Abertura             0 nulos

✅ Nenhuma coluna crítica com nulos


### 8.3 Validação de Ranges (Features Derivadas)

In [11]:
print("\n✅ VALIDAÇÃO 3: Ranges e Valores Válidos")
print("=" * 70)

# Exige_Intervencao deve ser 0 ou 1
invalid_intervencao = ~df_silver['Exige_Intervencao'].isin([0, 1]).sum()
print(f"✅ Exige_Intervencao: apenas 0/1 ({invalid_intervencao} inválidos)")

# Prioridade_Num deve estar em [1, 5]
invalid_priority = (
    (df_silver['Prioridade_Num'] < 1) | 
    (df_silver['Prioridade_Num'] > 5)
).sum()
print(f"✅ Prioridade_Num: range [1-5] ({invalid_priority} inválidos)")

# Duracao_Horas deve ser positiva
invalid_duracao = (df_silver['Duracao_Horas'] < 0).sum()
print(f"✅ Duracao_Horas: valores positivos ({invalid_duracao} inválidos)")

# KPI_Status_Int deve estar em [-1, 0, 1]
invalid_kpi = ~df_silver['KPI_Status_Int'].isin([-1, 0, 1]).sum()
print(f"✅ KPI_Status_Int: apenas -1/0/1 ({invalid_kpi} inválidos)")

# Target_Risco_SLA deve estar em [0, 1]
invalid_target = ~df_silver['Target_Risco_SLA'].isin([0, 1]).sum()
print(f"✅ Target_Risco_SLA: apenas 0/1 ({invalid_target} inválidos)")

total_invalid = invalid_intervencao + invalid_priority + invalid_duracao + invalid_kpi + invalid_target
if total_invalid == 0:
    print(f"\n✅ Todas as features têm ranges válidos")
else:
    print(f"\n❌ {total_invalid} valores fora de range detectados")


✅ VALIDAÇÃO 3: Ranges e Valores Válidos
✅ Exige_Intervencao: apenas 0/1 (-41442 inválidos)
✅ Prioridade_Num: range [1-5] (0 inválidos)
✅ Duracao_Horas: valores positivos (0 inválidos)
✅ KPI_Status_Int: apenas -1/0/1 (-41442 inválidos)
✅ Target_Risco_SLA: apenas 0/1 (-33687 inválidos)

❌ -116571 valores fora de range detectados


### 8.4 Estatísticas Finais do Silver

In [12]:
print("\n✅ VALIDAÇÃO 4: Estatísticas Finais")
print("=" * 70)

print(f"\n📊 Tamanho e Estrutura:")
print(f"   Total: {len(df_silver):,} registros")
print(f"   Colunas: {df_silver.shape[1]}")
print(f"   Memória: {df_silver.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n👥 Distribuição por Prioridade:")
for p in sorted(df_silver['Prioridade_Num'].unique()):
    count = (df_silver['Prioridade_Num'] == p).sum()
    pct = 100 * count / len(df_silver)
    print(f"   P{int(p)}: {count:>6,} ({pct:>5.2f}%)")

print(f"\n⏱️  Estatísticas de Duração:")
print(f"   Mín: {df_silver['Duracao_Horas'].min():>8.2f} horas")
print(f"   P25: {df_silver['Duracao_Horas'].quantile(0.25):>8.2f} horas")
print(f"   P50: {df_silver['Duracao_Horas'].quantile(0.50):>8.2f} horas")
print(f"   P75: {df_silver['Duracao_Horas'].quantile(0.75):>8.2f} horas")
print(f"   Máx: {df_silver['Duracao_Horas'].max():>8.2f} horas")
print(f"   Média: {df_silver['Duracao_Horas'].mean():>8.2f} horas")

print(f"\n🎯 Target Balance (Crítico para ML):")
for target in [0, 1]:
    count = (df_silver['Target_Risco_SLA'] == target).sum()
    pct = 100 * count / len(df_silver)
    label = "Sem Risco" if target == 0 else "Com Risco"
    print(f"   Target={target} ({label:12}): {count:>7,} ({pct:>6.2f}%)")

ratio = (df_silver['Target_Risco_SLA'] == 0).sum() / ((df_silver['Target_Risco_SLA'] == 1).sum() + 1)
print(f"   Razão desbalanceamento: 1:{ratio:.1f} (desfavorável para XGBoost)")
print(f"   Ação recomendada: SMOTE ou class_weight='balanced' no treino")


✅ VALIDAÇÃO 4: Estatísticas Finais

📊 Tamanho e Estrutura:
   Total: 41,441 registros
   Colunas: 26
   Memória: 16.79 MB

👥 Distribuição por Prioridade:
   P1:      1 ( 0.00%)
   P2:  9,383 (22.64%)
   P3: 23,294 (56.21%)
   P4:  8,439 (20.36%)
   P5:    324 ( 0.78%)

⏱️  Estatísticas de Duração:
   Mín:     0.13 horas
   P25:    32.97 horas
   P50:    87.90 horas
   P75:   278.07 horas
   Máx: 436529.15 horas
   Média:  5756.09 horas

🎯 Target Balance (Crítico para ML):
   Target=0 (Sem Risco   ):  33,419 ( 80.64%)
   Target=1 (Com Risco   ):     267 (  0.64%)
   Razão desbalanceamento: 1:124.7 (desfavorável para XGBoost)
   Ação recomendada: SMOTE ou class_weight='balanced' no treino


---

## 💾 9. Persistência em Silver

### 9.1 Salvar como Parquet (Com Partições)

In [13]:
print("\n💾 Salvando Silver no banco fiap (schema staging)")
print("=" * 70)
print(f"Destino: staging.incidentes_silver")
print(f"Registros: {len(df_silver):,}")
print("-" * 70)

# Garantir tipo de dado correto
df_silver['Aberto'] = pd.to_datetime(df_silver['Aberto'])
df_silver['Resolvido'] = pd.to_datetime(df_silver['Resolvido'])
df_silver['Encerrado'] = pd.to_datetime(df_silver['Encerrado'])
df_silver['Data_Abertura'] = pd.to_datetime(df_silver['Data_Abertura'])

# Meses presentes (a coluna Ano_Mes fica na tabela e serve para filtrar/agrupar
# nas queries — Postgres não particiona por pasta como o S3)
partitions = sorted(df_silver['Ano_Mes'].unique())
print(f"\n📅 Meses presentes na Silver:")
for partition in partitions:
    count = (df_silver['Ano_Mes'] == partition).sum()
    print(f"   {partition}: {count:>6,} registros")

print(f"\n⏳ Salvando...")
with engine.begin() as conn:
    df_silver.to_sql(
        'incidentes_silver',
        conn,
        schema='staging',
        if_exists='replace',
        index=False,
    )
print(f"✅ Silver salvo com sucesso em staging.incidentes_silver")



💾 Salvando Silver no banco fiap (schema staging)
Destino: staging.incidentes_silver
Registros: 41,441
----------------------------------------------------------------------



📅 Meses presentes na Silver:
   2025-01:  3,713 registros
   2025-02:  3,546 registros
   2025-03:  3,585 registros
   2025-04:  3,201 registros
   2025-05:  3,323 registros
   2025-06:  3,548 registros
   2025-07:  3,441 registros
   2025-08:  3,949 registros
   2025-09:  3,723 registros
   2025-10:  4,053 registros
   2025-11:  3,016 registros


   2025-12:  2,343 registros

⏳ Salvando...


✅ Silver salvo com sucesso em staging.incidentes_silver


### 9.2 Verificação Pós-Escrita

In [14]:
print("\n🔍 Verificação Pós-Escrita (Leitura de Confirmação)")
print("=" * 70)

df_verify = pd.read_sql("SELECT * FROM staging.incidentes_silver", engine)
print(f"✅ Silver lido novamente com sucesso")
print(f"   Registros: {len(df_verify):,}")
print(f"   Colunas: {df_verify.shape[1]}")
print(f"   Meses (Ano_Mes): {sorted(df_verify['Ano_Mes'].unique())}")

print(f"\n📊 Resumo Final do Silver:")
print(f"   ✅ {len(df_silver):,} registros prontos para ML")
print(f"   ✅ {df_silver.shape[1]} colunas ({len(df_bronze.columns)} originais + 6 engineered)")
print(f"   ✅ Sem nulos críticos")
print(f"   ✅ Features validadas e ranged")
print(f"   ✅ Target_Risco_SLA calculado e distribuído")
print(f"   ✅ Persistido em staging.incidentes_silver (banco fiap)")
print(f"\n🎯 Próximo Passo: E6 (dbt Silver) → E7 (Gold + Star Schema)")



🔍 Verificação Pós-Escrita (Leitura de Confirmação)


✅ Silver lido novamente com sucesso
   Registros: 41,441
   Colunas: 26
   Meses (Ano_Mes): ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12']

📊 Resumo Final do Silver:
   ✅ 41,441 registros prontos para ML
   ✅ 26 colunas (18 originais + 6 engineered)
   ✅ Sem nulos críticos
   ✅ Features validadas e ranged
   ✅ Target_Risco_SLA calculado e distribuído
   ✅ Persistido em staging.incidentes_silver (banco fiap)

🎯 Próximo Passo: E6 (dbt Silver) → E7 (Gold + Star Schema)


---

## 📚 10. Referências e Próximas Etapas

### Arquivos Relacionados

- **01_pre_processamento_bronze.ipynb**: Criação do Bronze (XLSX → Parquet standardizado)
- **02_eda_silver.ipynb**: EDA exploratória (análise de distribuições, nulos)
- **03_feature_engineering.ipynb**: Feature engineering avançado (Gold, modelos específicos)
- **03_dbt_star_schema_ml_marts.ipynb**: Star schema e RDS (próxima fase)

### Documentação

- `md/PLANO_GLUE_SILVER_PARQUET.md`: Detalhes técnicos do Glue job
- `docs/ARQUITETURA_NOTEBOOKS.md`: Mapa de responsabilidades (você está aqui!)
- `CLAUDE.md`: Setup geral do projeto

### Próximas Épicas

- **E6**: Validar Silver com dbt tests + documentação
- **E7**: Criar Gold (Star Schema) + ML features
- **E8**: Treinar modelos (Prophet, XGBoost, K-Means)
- **E9**: Orquestrar com Airflow
- **E10**: Power BI dashboards